# Training Rectified Flow for Data-to-Data Interpolation

This notebook implements a **training pipeline** for Rectified Flow on FashionMNIST data-to-data tasks

## Core Concept

A Rectified Flow learns a neural ODE velocity field that transports a source distribution to a target distribution. We optimize:

$$\mathcal{L} = \mathbb{E}_{t \sim U[0,1], x_a, x_b} \left[ \left\| v_\theta(z_t, t) - (x_b - x_a) \right\|^2 \right]$$

where:
- $z_t = (1-t) x_a + t x_b$ (straight-line path from source to target)
- $v_\theta$ is the learned velocity field (neural network)
- Target velocity is $(x_b - x_a)$

## Key Properties

1. **Straight-line interpolation**: Data points move linearly from source to target
2. **Velocity matching**: Network learns to predict exact endpoint transformation
3. **ODE sampling**: At test time, integrate velocity forward to generate trajectories


## Prerequisite: Colab Setup

Run all cells in [`00_colab_setup.ipynb`]first! Notebook 00 owns repository cloning, working-directory setup, and the Python import path. This notebook assumes that setup is complete!

In [1]:
# check environment
import os
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')

Device: cuda
GPU: Tesla T4
CUDA: 13.0


In [5]:
# core imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json
from dataclasses import dataclass, field
from typing import Dict, Optional, List

import sys
from pathlib import Path

repo_path = "/content/minRF_interpolation"
assert Path(repo_path, "api", "flow_model_api.py").is_file()

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# use project APIs
from api.flow_model_api import (
    DatasetSpec,
    ModelSpec,
    DataToDataRF,
    build_dataset_loaders,
    build_flow_model,
    save_checkpoint,
    load_checkpoint,
)
from api.training_api import TrainingSpec, checkpoint_paths, train_data_to_data_flow

print('Imports successful')

Imports successful


In [6]:
# disable Weights & Biases (for clean training without external logging)
os.environ['WANDB_DISABLED'] = 'true'
print('W&B disabled')

W&B disabled


## Training Configuration


### FashionMNIST Dataset Overview

FashionMNIST contains 10 classes of clothing items. Default configuration trains on:

| Class ID | Name | Role in Training |
|----------|------|------------------|
| 0 | T-shirt/top | - |
| 1 | Trouser | - |
| 2 | Pullover | - |
| 3 | Dress | - |
| 4 | Coat | - |
| 5 | Sandal | - |
| 6 | Shirt | - |
| **7** | **Sneaker** | **Source (x_a)** |
| 8 | Bag | - |
| **9** | **Ankle Boot** | **Target (x_b)** |

**Interpolation Task**: Model learns to transport images from Sneaker class to Ankle Boot class.

This pairing was chosen because both are footwear with distinct visual features, making it a meaningful semantic transformation task


In [7]:
config = TrainingSpec(
    model_profile='medium',
    source_class=7,
    target_class=9,
    train_epochs=100,
    model_name='fmnist_d2d_training',
    load_best_if_available=False,
    checkpoint_every=20,  # set to 25 for checkpoints at epochs 25, 50, ...
    mixed_precision=True,  # CUDA FP16 + gradient scaling, suitable for a T4
    resume_from=None,  # set to the latest or a numbered checkpoint to continue
)
output_dir = Path('contents')
print('Configuration created')
print(f'  Source class: {config.source_class} (Sneaker)')
print(f'  Target class: {config.target_class} (Ankle Boot)')
print(f'  Model: {config.model_profile}')
print(f'  Epochs (total target, including resumed epochs): {config.train_epochs}')

Configuration created
  Source class: 7 (Sneaker)
  Target class: 9 (Ankle Boot)
  Model: medium
  Epochs: 100


## Data Loading

In [8]:
print('The training API builds FashionMNIST loaders once when training starts.')
print(f'Class pair: {config.source_class} -> {config.target_class}')
print(f'Batch size: {config.batch_size} train / {config.test_batch_size} validation')

100%|██████████| 26.4M/26.4M [00:01<00:00, 19.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 306kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.63MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.1MB/s]


Train loader: 46 batches
Test loader: 8 batches
Total train samples: 60000


## Model Initialization

In [9]:
print(f'Model profile: {config.model_profile}')
print('The training API constructs the model and optimizer via flow_model_api.')
print(f'CUDA mixed precision requested: {config.mixed_precision}')

Model parameters: 2,393,796
Trainable parameters: 2,393,796
Model profile: medium


## Training Loop

For each epoch:
1. **Forward pass**: $z_t = (1-t) x_a + t x_b$, predict $v_\theta(z_t, t)$
2. **Loss**: MSE between predicted and target velocity
3. **Backward pass**: Update model parameters
4. **Validation**: Evaluate on test set, track best model
5. **Early stopping**: Stop if validation loss plateaus


In [10]:
print('Training entry point:', train_data_to_data_flow.__module__)
print(f'Numbered checkpoint interval: {config.checkpoint_every} epochs')

Training functions defined


In [11]:
# prepare checkpoint directory
checkpoint_dir = Path(config.checkpoint_dir)
output_dir.mkdir(parents=True, exist_ok=True)

paths = checkpoint_paths(config.checkpoint_dir, config.model_name)
checkpoint_latest = paths['latest']
checkpoint_best = paths['best']

print(f'Checkpoints will be saved to: {checkpoint_dir}')

Checkpoints will be saved to: assets/fmnist_d2d_runs


In [ ]:
result = train_data_to_data_flow(spec=config, device=device)
model = result['model']
flow_model = result['rf']
optimizer = result['optimizer']
train_loader = result['train_loader']
test_loader = result['test_loader']
model_spec = result['setup']['model_spec']
history = {'train_loss': result['train_history'], 'test_loss': result['val_history']}
best_test_loss = result['best_val_loss']
epoch_with_best = result['best_epoch'] - 1
checkpoint_latest = result['latest_path']
checkpoint_best = result['best_path']
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Best test loss: {best_test_loss:.6f} at epoch {epoch_with_best + 1}')

Starting training for 100 epochs...

Epoch   1 | Train loss: 0.305650 | Test loss: 0.196043 | Best: 0.196043 @ 1 | Patience: 0/10
Epoch  10 | Train loss: 0.069720 | Test loss: 0.064376 | Best: 0.064376 @ 10 | Patience: 0/10
Epoch  20 | Train loss: 0.055450 | Test loss: 0.051872 | Best: 0.051741 @ 19 | Patience: 1/10
Epoch  30 | Train loss: 0.048807 | Test loss: 0.049111 | Best: 0.047645 @ 29 | Patience: 1/10
Epoch  40 | Train loss: 0.047419 | Test loss: 0.046026 | Best: 0.045816 @ 37 | Patience: 3/10


## Training Summary

In [ ]:
import matplotlib.pyplot as plt

# plot training curves
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss', linewidth=2)
plt.plot(history['test_loss'], label='Test Loss', linewidth=2)
plt.axvline(x=epoch_with_best, color='red', linestyle='--', label=f'Best epoch ({epoch_with_best+1})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Rectified Flow Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150)
plt.show()

print(f'Training curve saved')

In [ ]:
# save training history
history_file = output_dir / 'training_history.json'
with open(history_file, 'w') as f:
    json.dump(history, f, indent=2)

print(f'History saved to {history_file}')

In [ ]:
# print final statistics
print('\n' + '='*60)
print('TRAINING COMPLETED')
print('='*60)
print(f'Total epochs trained: {len(history["train_loss"])}')
print(f'Best epoch: {epoch_with_best + 1}')
print(f'Best test loss: {best_test_loss:.6f}')
print(f'Final train loss: {history["train_loss"][-1]:.6f}')
print(f'Final test loss: {history["test_loss"][-1]:.6f}')
print(f'\nCheckpoint saved: {checkpoint_best}')
print('='*60)

## Load Best Model for Evaluation


In [ ]:
# load best checkpoint for subsequent evaluation
# recreate model and optimizer
best_model, _, best_optimizer = build_flow_model(model_spec)
best_model = best_model.to(device)

# load checkpoint state
checkpoint = load_checkpoint(
    best_model,
    best_optimizer,
    checkpoint_best,
    device=device
)

best_flow_model = DataToDataRF(best_model)

print(f'Best model loaded from epoch {epoch_with_best + 1}')
print(f'Test loss at that epoch: {best_test_loss:.6f}')